This will create a tool calling agent, if you give an url of youtube it will extract the transcripts, create summary

recursive tool calling also this will ai agents to dynamicaly navigate whcih tools to use and when to use them

Objective
-> Build both manual and automated tool calling
-> design flexible workflow that allows LLMs to reason about when and how to use tools

pip install yt-dlp youtube-transcript-api langchain-community langchain langchain-openai


In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
import re

In [6]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [7]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11-character YouTube video ID from a URL.

Args:
    url (str): A YouTube URL containing a video ID.

Returns:
    str: Extracted video ID or error message if parsing fails.
----------------------------
<function extract_video_id at 0x0000026D513E72E0>


In [8]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [9]:
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x0000026D513E72E0>)

In [10]:
tools = []
tools.append(extract_video_id)

In [11]:
from youtube_transcript_api import YouTubeTranscriptApi


@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [13]:
tools.append(fetch_transcript)

In [26]:
from langchain.tools import tool
from typing import List, Dict, Union
from yt_dlp import YoutubeDL


@tool
def search_youtube(query: str, max_results: int = 10) -> Union[List[Dict[str, str]], str]:
    """
    Search YouTube for videos matching the query (uses yt-dlp, which is actively maintained).

    Args:
        query (str): The search term to look for on YouTube.
        max_results (int): Maximum number of results to return (default 10).

    Returns:
        A list of dicts: [{'title': ..., 'video_id': ..., 'url': ...}, ...]
        or an error string if the search fails.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "extract_flat": True,   # don't fetch each video's full metadata
        "default_search": "ytsearch",
    }
    try:
        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(f"ytsearch{max_results}:{query}", download=False)

        entries = info.get("entries", []) or []
        results = []
        for e in entries:
            if not e:
                continue
            vid = e.get("id")
            if not vid:
                continue
            results.append({
                "title": e.get("title", ""),
                "video_id": vid,
                "url": e.get("url") or f"https://youtu.be/{vid}",
            })
        return results
    except Exception as e:
        return f"Error: {str(e)}"

In [40]:
import json
from IPython.display import display, JSON

search_out = search_youtube.run("Generative AI")

# Interactive tree view (VS Code shows it if its JSON renderer is enabled)
display(JSON(search_out, expanded=True))

# Always-readable fallback
print(json.dumps(search_out, indent=2, ensure_ascii=False))

<IPython.core.display.JSON object>

[
  {
    "title": "Generative AI Explained In 5 Minutes | What Is GenAI? | Introduction To Generative AI | Simplilearn",
    "video_id": "NRmAXDWJVnU",
    "url": "https://www.youtube.com/watch?v=NRmAXDWJVnU"
  },
  {
    "title": "AI, Machine Learning, Deep Learning and Generative AI Explained",
    "video_id": "qYNweeDHiyU",
    "url": "https://www.youtube.com/watch?v=qYNweeDHiyU"
  },
  {
    "title": "Generative AI in a Nutshell - how to survive and thrive in the age of AI",
    "video_id": "2IK3DFHRFfw",
    "url": "https://www.youtube.com/watch?v=2IK3DFHRFfw"
  },
  {
    "title": "Gen AI Course | Gen AI Tutorial For Beginners",
    "video_id": "d4yCWBGFCEs",
    "url": "https://www.youtube.com/watch?v=d4yCWBGFCEs"
  },
  {
    "title": "What is generative AI and how does it work? – The Turing Lectures with Mirella Lapata",
    "video_id": "_6R7Ym6Vy_I",
    "url": "https://www.youtube.com/watch?v=_6R7Ym6Vy_I"
  },
  {
    "title": "GenAI Essentials – Full Course for Beginners",

In [29]:
tools.append(search_youtube)
print("Registered tools:", [t.name for t in tools])

Registered tools: ['extract_video_id', 'fetch_transcript', 'search_youtube']


In [32]:
import logging
import yt_dlp


class _YtDlpSilentLogger:
    """Silences yt-dlp's noisy stdout/stderr output."""
    def debug(self, msg): pass
    def info(self, msg): pass
    def warning(self, msg): pass
    def error(self, msg):
        logging.getLogger("yt_dlp").error(msg)


@tool
def get_full_metadata(url: str) -> dict:
    """
    Extract metadata from a YouTube URL, including title, views, duration,
    channel, likes, comments, and chapters.

    Args:
        url (str): A YouTube video URL.

    Returns:
        dict: Metadata fields, or {'error': ...} on failure.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "logger": _YtDlpSilentLogger(),
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
        return {
            "title": info.get("title"),
            "views": info.get("view_count"),
            "duration": info.get("duration"),
            "channel": info.get("uploader"),
            "likes": info.get("like_count"),
            "comments": info.get("comment_count"),
            "chapters": info.get("chapters", []) or [],
        }
    except Exception as e:
        return {"error": str(e)}

In [41]:
meta_data = get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")

display(JSON(meta_data, expanded=True))
print(json.dumps(meta_data, indent=2, ensure_ascii=False, default=str))

<IPython.core.display.JSON object>

{
  "title": "What is Retrieval-Augmented Generation (RAG)?",
  "views": 1864250,
  "duration": 395,
  "channel": "IBM Technology",
  "likes": 43696,
  "comments": 979,
  "chapters": [
    {
      "start_time": 0.0,
      "title": "Introduction",
      "end_time": 18.0
    },
    {
      "start_time": 18.0,
      "title": "What is RAG",
      "end_time": 42.0
    },
    {
      "start_time": 42.0,
      "title": "An anecdote",
      "end_time": 82.0
    },
    {
      "start_time": 82.0,
      "title": "Two problems",
      "end_time": 138.0
    },
    {
      "start_time": 138.0,
      "title": "Large language models",
      "end_time": 263.0
    },
    {
      "start_time": 263.0,
      "title": "How does RAG help",
      "end_time": 395
    }
  ]
}


In [35]:
tools.append(get_full_metadata)

In [42]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.

    Args:
        url (str): YouTube video URL (any format).

    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order.
    """
    ydl_opts = {
        "quiet": True,
        "no_warnings": True,
        "skip_download": True,
        "logger": _YtDlpSilentLogger(),
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)

        thumbnails = []
        for t in info.get("thumbnails", []) or []:
            if "url" not in t:
                continue
            w, h = t.get("width"), t.get("height")
            resolution = f"{w}x{h}" if w and h else ""
            thumbnails.append({
                "url": t["url"],
                "width": w,
                "height": h,
                "resolution": resolution,
            })
        return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [43]:
thumbnails = get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails, expanded=True))
print(json.dumps(thumbnails, indent=2, ensure_ascii=False, default=str))

<IPython.core.display.JSON object>

[
  {
    "url": "https://i.ytimg.com/vi/qWHaMrR5WHQ/3.jpg",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi_webp/qWHaMrR5WHQ/3.webp",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi/qWHaMrR5WHQ/2.jpg",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi_webp/qWHaMrR5WHQ/2.webp",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi/qWHaMrR5WHQ/1.jpg",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi_webp/qWHaMrR5WHQ/1.webp",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi/qWHaMrR5WHQ/mq3.jpg",
    "width": null,
    "height": null,
    "resolution": ""
  },
  {
    "url": "https://i.ytimg.com/vi_webp/qWHaMrR5WHQ/mq3.webp",
    "width": null,
    "height": null,
    

In [44]:
tools.append(get_thumbnails)

In [45]:
tools

[StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x0000026D513E72E0>),
 StructuredTool(name='fetch_transcript', description='Fetches the transcript of a YouTube video.\n\nArgs:\n    video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").\n    language (str): Language code for the transcript (e.g., "en", "es").\n\nReturns:\n    str: The transcript text or an error message.', args_schema=<class 'langchain_core.utils.pydantic.fetch_transcript'>, func=<function fetch_transcript at 0x0000026D528F6F20>),
 StructuredTool(name='search_youtube', description="Search YouTube for videos matching the query (uses yt-dlp, which is actively maintained).\n\nArgs:\n    query (str): The search term to

In [48]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
model="deepseek/deepseek-v4-flash"
llm = init_chat_model(
    model=model,
    model_provider="openrouter",
)

In [50]:
response = llm.invoke("Hello, how are you?")  

In [51]:
response

AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm running perfectly and ready to help you. How can I assist you today?", additional_kwargs={}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-20260423', 'id': 'gen-1781787867-XW9wthIfBHhON5QElnrs', 'created': 1781787867, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.12e-05, 'cost_details': {'upstream_inference_completions_cost': 9.8e-06, 'upstream_inference_prompt_cost': 1.4e-06, 'upstream_inference_cost': 1.12e-05}}, id='lc_run--019edad5-385b-70c0-a4a4-707b25d7b519-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 35, 'total_tokens': 45, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

In [52]:
llm_with_tools = llm.bind_tools(tools)

In [53]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))

C:\Users\sku104\AppData\Local\Temp\ipykernel_11852\519310222.py:5: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "parameters": tool.args_schema.schema() if tool.args_schema else {},


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [54]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


In [67]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

In [57]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]


In [58]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'reasoning_content': 'Let me extract the video ID first, then fetch the transcript and metadata.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Let me extract the video ID first, then fetch the transcript and metadata.'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-20260423', 'id': 'gen-1781789584-L6A5k7MlrJPZaxboBRXD', 'created': 1781789584, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 8.2724e-05, 'cost_details': {'upstream_inference_completions_cost': 1.4896e-05, 'upstream_inference_prompt_cost': 6.7828e-05, 'upstream_inference_cost': 8.2724e-05}, 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402'}, id='lc_run--019edaef-6a02-7c50-854a-1a45559ed14f-0', tool_calls=[{'name': 'extract_video_id', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'call_00_GAI6gprJltxbMdGmkvng88

In [59]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [60]:
response_1.tool_calls

[{'name': 'extract_video_id',
  'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'},
  'id': 'call_00_GAI6gprJltxbMdGmkvng8838',
  'type': 'tool_call'}]

In [61]:
tool_calls_1 = response_1.tool_calls
tool_name=tool_calls_1[0]['name']
print(tool_name)

extract_video_id


In [62]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

call_00_GAI6gprJltxbMdGmkvng8838


In [63]:
my_tool=tool_mapping[tool_calls_1[0]['name']]

In [64]:
my_tool

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x0000026D513E72E0>)

In [65]:
video_id =my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [68]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

In [69]:
messages

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='T-D1OfcDW1M', tool_call_id='call_00_GAI6gprJltxbMdGmkvng8838')]

In [73]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'reasoning_content': 'The video ID extracted is T-D1OfcDW1M. Now let me fetch the transcript in English.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The video ID extracted is T-D1OfcDW1M. Now let me fetch the transcript in English.'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-20260423', 'id': 'gen-1781790518-H6d1mR9tYJo51Ny9rPpn', 'created': 1781790518, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.00010878, 'cost_details': {'upstream_inference_completions_cost': 1.8228e-05, 'upstream_inference_prompt_cost': 9.0552e-05, 'upstream_inference_cost': 0.00010878}}, id='lc_run--019edafd-aa00-7f01-a839-2ea06ef1dffc-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'}, 'id': 'call_0a0a23e258ca4fea8c365f0c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [75]:
messages.append(response_2)

In [76]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'call_0a0a23e258ca4fea8c365f0c',
  'type': 'tool_call'}]

In [77]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

In [78]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

In [79]:
summary = llm_with_tools.invoke(messages)

In [81]:
summary.content

'## 📌 Summary: "Retrieval-Augmented Generation (RAG)" — IBM Technology\n\n**Presenter:** Marina Danilevsky — Senior Research Scientist at IBM Research\n\n---\n\n### 🎯 Main Topic\nThe video explains **Retrieval-Augmented Generation (RAG)**, a framework that improves large language models (LLMs) by making them **more accurate** and **up-to-date**.\n\n---\n\n### 🧩 Key Concepts\n\n#### 1. **Two Core Problems with LLMs**\n- **No source / Hallucination** — LLMs confidently produce answers without citing evidence.\n- **Outdated information** — LLMs are trained on static data and cannot naturally update without retraining.\n\n#### 2. **The RAG Solution**\nInstead of relying *only* on what the LLM learned during training, RAG introduces a **content store** (e.g., the internet, a company\'s policy documents, etc.) that the model queries *before* generating an answer.\n\n**How it works:**\n1. User submits a **prompt** (question).\n2. The LLM **retrieves** relevant content from the data store.\n3.

In [82]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

In [83]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [84]:
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)

In [85]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 # Summary: **What is LangChain?** — IBM Technology

**⏱ Duration:** 8 min 7 sec | **👁 Views:** 724K | **👍 Likes:** 17K | **📺 Channel:** IBM Technology

---

## 🧠 What is LangChain?

LangChain is an **open-source orchestration framework** (available in Python and JavaScript) for building applications that use Large Language Models (LLMs). Created by **Harrison Chase** and launched in October 2022, it became the fastest-growing open-source project on GitHub by June 2023.

The core idea: Use **abstractions** to simplify LLM programming — just like a thermostat abstracts away complex circuitry, LangChain abstracts common LLM workflow steps so you can chain them together with minimal code.

---

## 🧩 Key Components

| Component | What It Does |
|-----------|-------------|
| **LLM Module** | Provides a standard interface for nearly any LLM — closed-source (GPT-4) or open-source (Llama 2) — using just an API key. |
| **Prompts** | Formalizes prompt composition via templates (e

---
Up to this point, you've demonstrated how to manually orchestrate the tool calling process step by step. You first invoked the LLM with the user's query, interpreted its decision to use the `extract_video_id` tool, executed that tool, fed the result back to the LLM, processed its next decision to use the `fetch_transcript` tool, executed that tool, and finally had the LLM generate a summary based on the transcript.

Now you'll see how to accomplish the same workflow more efficiently using LangChain's chain functionality, which automates this back-and-forth process of tool selection, execution, and response handling.


In [86]:
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

In [87]:
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

In [88]:
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
)

In [89]:
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

In [90]:
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
)

In [91]:
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"])

In [95]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [96]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = summarization_chain.invoke(query)
print("Video Summary:\n", result)

Video Summary:
 # 📺 Video Summary: *What is Retrieval-Augmented Generation (RAG)?*

**Channel:** IBM Technology | **Duration:** ~6 min 35 sec | **Views:** 1.86M | **Likes:** 43.7K

---

## 🧠 Overview

Presented by **Marina Danilevsky**, a Senior Research Scientist at IBM Research, this video explains **Retrieval-Augmented Generation (RAG)** — a framework designed to make large language models (LLMs) more **accurate** and **up-to-date**.

---

## 🎯 Key Points

### 1️⃣ The Problem with LLMs (The "Generation" Part)
The video opens with an anecdote: the speaker's kids ask **"What planet has the most moons?"** She confidently answers **Jupiter (88 moons)** — but this is **wrong and outdated**. This mirrors two critical LLM challenges:

| Problem | Description |
|---|---|
| ❌ **No Source** | LLMs produce answers confidently without citing evidence, making them prone to **hallucinations**. |
| ❌ **Outdated Info** | LLMs are frozen at the time of training. New discoveries (e.g., **Saturn now h

In [98]:
query = {"query": "Get top 3 youtube videos in India and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Video Summary:
 Let me fetch the metadata for the top 3 videos from the India trending search.


In [99]:
result

'Let me fetch the metadata for the top 3 videos from the India trending search.'

In [100]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]


def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))


def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [101]:
query_us = {"query": "Show top 3 US trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nUS Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching US trending videos:", e)


US Trending Videos:
 content='Here are **3 notable trending YouTube videos in the US** right now, with their metadata and thumbnails:\n\n---\n\n## 🔥 #1 — 2025 TikTok Rewind 🔥 Most Viral Funny Videos of the Year\n| Detail | Value |\n|--------|-------|\n| **Channel** | America\'s Funniest Home Videos |\n| **Views** | **1,541,809** 👀 |\n| **Duration** | 23 min 25 sec |\n| **Likes** | 9,010 |\n| **Comments** | 283 |\n| **URL** | [Watch on YouTube](https://www.youtube.com/watch?v=KldVQBAkjuo) |\n\n**Thumbnail (max res):**\n![Thumbnail](https://i.ytimg.com/vi/KldVQBAkjuo/maxresdefault.jpg)\n\n---\n\n## 🔥 #2 — Amazing Inventions That Are on Another Level | Best of 2025 So Far!\n| Detail | Value |\n|--------|-------|\n| **Channel** | TechTrends |\n| **Views** | **287,514** 👀 |\n| **Duration** | 22 min 46 sec |\n| **Likes** | 1,774 |\n| **Comments** | 90 |\n| **URL** | [Watch on YouTube](https://www.youtube.com/watch?v=MmgRnR0hGN8) |\n\n**Thumbnail (max res):**\n![Thumbnail](https://i.ytimg.co